# iiq2img — Usage Examples

Fast IIQ-to-image converter for Phase One iXM-GS120 (120MP) raw files with georeferencing support.

In [1]:
from iiq2img import (
    convert_iiq,
    batch_convert,
    extract_metadata,
    extract_geo_info,
)
from pathlib import Path

In [2]:
input_dir = Path("PhaseOneSample")
output_dir = Path().cwd() / "output"

In [3]:
input_files = list(input_dir.glob("*.IIQ"))
first_file = input_files[0]

## Single File Conversion

In [4]:
result = convert_iiq(first_file, output_dir / (first_file.stem + ".jpg"))

print(f"Output:     {result.output_path}")
print(f"Resolution: {result.width}x{result.height}")
print(f"Time:       {result.elapsed_ms:.0f}ms")
print(f"Size:       {result.file_size_bytes / 1024 / 1024:.1f} MB")

Output:     /home/nick/Documents/Work code/SKW_RAW/output/P0286669.jpg
Resolution: 12768x9564
Time:       3364ms
Size:       54.5 MB


## Bulk Conversion

In [ ]:
results = batch_convert(
    input_dir=input_dir,
    output_dir=output_dir,
    output_format="jpg",
    compress_quality=90,
    workers=8,
)

  0%|          | 0/70 [00:00<?, ?img/s]

## Georeferencing

Adds spatial reference using GPS/IMU data from IIQ metadata.
- **JPEG/PNG**: sidecar world file (.jgw/.pgw)
- **TIFF**: embedded GeoTIFF with CRS and affine transform

In [ ]:
# JPEG with world file
result = convert_iiq(first_file, output_dir / (first_file.stem + ".jpg"), georef=True)

In [ ]:
# GeoTIFF with embedded CRS
result = convert_iiq(first_file, output_dir / (first_file.stem + ".tif"), georef=True)

In [ ]:
# GSD and footprint from metadata
meta = extract_metadata("PhaseOneSample/P0286625.IIQ")
geo = extract_geo_info(
    meta["xmp"], focal_length_mm=80.0, image_width=12768, image_height=9564
)

print(f"Position:  {geo.latitude:.6f}, {geo.longitude:.6f}")
print(f"Altitude:  {geo.altitude_agl:.1f}m AGL")
print(f"GSD:       {geo.gsd * 100:.2f} cm/pixel")
print(f"Footprint: {geo.footprint_width:.1f}m x {geo.footprint_height:.1f}m")

Position:  -34.560007, 118.378565
Altitude:  25.8m AGL
GSD:       0.11 cm/pixel
Footprint: 14.2m x 10.6m
